# Text Splitters & Chunking

This notebook explores how large documents are divided into
smaller chunks for use in a RAG pipeline.

Topics:
- Why chunking is required
- Chunk size
- Chunk overlap
- Recursive character splitting
- Inspecting generated chunks
- Comparing different chunk configurations

In [3]:
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = Path("../documents/hr_policy.pdf")

loader = PyPDFLoader(str(PDF_PATH))
documents = loader.load()

print(f"Loaded documents: {len(documents)}")

Loaded documents: 16


In [33]:
from langchain_text_splitters import CharacterTextSplitter

fixed_splitter = CharacterTextSplitter(
    separator="",
    chunk_size=50,
    chunk_overlap=0,
)

documents = loader.load()

fixed_chunks = fixed_splitter.split_documents(documents)

print(f"Documents: {len(documents)}")
print(f"Chunks: {len(fixed_chunks)}")
for i, chunk in enumerate(fixed_chunks[:3]):
    print(f"\n--- Chunk {i} ---")
    print(chunk.page_content)

Documents: 16
Chunks: 563

--- Chunk 0 ---
HR, Leave and Attendance Policy 
Version – 1.5.2

--- Chunk 1 ---
Effective From:1stFebruary,2023

--- Chunk 2 ---
This document contains trade secrets and


In [32]:
for i, chunk in enumerate(fixed_chunks[:5]):
    print(f"\n{'=' * 60}")
    print(f"CHUNK {i}")
    print(f"{'=' * 60}")
    print(chunk.page_content)
    print(f"\nLength: {len(chunk.page_content)}")
    print(f"Metadata: {chunk.metadata}")


CHUNK 0
HR, Leave and Attendance Policy 
Version – 1.5.2

Length: 48
Metadata: {'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2023-04-13T09:38:48+00:00', 'author': 'Invx28', 'moddate': '2023-04-13T15:09:50+05:30', 'source': '../documents/hr_policy.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1'}

CHUNK 1
Effective From:1stFebruary,2023

Length: 31
Metadata: {'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2023-04-13T09:38:48+00:00', 'author': 'Invx28', 'moddate': '2023-04-13T15:09:50+05:30', 'source': '../documents/hr_policy.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1'}

CHUNK 2
This document contains trade secrets and

Length: 40
Metadata: {'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2023-04-13T09:38:48+00:00', 'author': 'Invx28', 'moddate': '2023-04-13T15:09:50+05:30', 'source': '../documents/hr_policy.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1'}

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = text_splitter.split_documents(documents)

print(f"Documents: {len(documents)}")
print(f"Chunks: {len(chunks)}")

Documents: 16
Chunks: 70


In [7]:
print(chunks[0].page_content)
print(chunks[0].metadata)

HR, Leave and Attendance Policy 
Version – 1.5.2 
Effective From:1stFebruary,2023 
 
 
 
 
 
 
 
 
 
 
 
 
 
This document contains trade secrets and proprietary information  of Innvonix Tech Solutions Private Limited. Any Use, Disclosure, dissemination,  forwarding, 
printing or copying of this publication is absolutely prohibited without the express written permission of Innvonix Tech Solu tions Private Limited ©2023.If you are
{'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2023-04-13T09:38:48+00:00', 'author': 'Invx28', 'moddate': '2023-04-13T15:09:50+05:30', 'source': '../documents/hr_policy.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1'}


In [20]:
for i, chunk in enumerate(chunks[:5]):
    print(f"\n{'=' * 60}")
    print(f"CHUNK {i}")
    print(f"{'=' * 60}")
    print(chunk.page_content)
    print(f"\nMetadata: {chunk.metadata}")


CHUNK 0
HR, Leave and Attendance Policy 
Version – 1.5.2 
Effective From:1stFebruary,2023 
 
 
 
 
 
 
 
 
 
 
 
 
 
This document contains trade secrets and proprietary information  of Innvonix Tech Solutions Private Limited. Any Use, Disclosure, dissemination,  forwarding, 
printing or copying of this publication is absolutely prohibited without the express written permission of Innvonix Tech Solu tions Private Limited ©2023.If you are

Metadata: {'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2023-04-13T09:38:48+00:00', 'author': 'Invx28', 'moddate': '2023-04-13T15:09:50+05:30', 'source': '../documents/hr_policy.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1'}

CHUNK 1
This document contains trade secrets and proprietary information  of Innvonix Tech Solutions Private Limited. Any Use, Disclosure, dissemination,  forwarding, 
printing or copying of this publication is absolutely prohibited without the express written permission of Innvonix

In [13]:
chunk_lengths = [len(chunk.page_content) for chunk in chunks]

print(f"Number of chunks: {len(chunk_lengths)}")
print(f"Minimum: {min(chunk_lengths)} characters")
print(f"Maximum: {max(chunk_lengths)} characters")
print(f"Average: {sum(chunk_lengths) / len(chunk_lengths):.2f} characters")

Number of chunks: 70
Minimum: 118 characters
Maximum: 499 characters
Average: 400.46 characters


In [22]:
small_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,
)

small_chunks = small_splitter.split_documents(documents)

print(f"Chunks with size 200: {len(small_chunks)}")
print(f"500-size chunks: {len(chunks)}")
print(f"200-size chunks: {len(small_chunks)}")

Chunks with size 200: 183
500-size chunks: 70
200-size chunks: 183


In [25]:
large_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
)

large_chunks = large_splitter.split_documents(documents)

print(f"Chunks with size 1000: {len(large_chunks)}")
print(f"200:  {len(small_chunks)} chunks")
print(f"500:  {len(chunks)} chunks")
print(f"1000: {len(large_chunks)} chunks")

Chunks with size 1000: 37
200:  183 chunks
500:  70 chunks
1000: 37 chunks


In [26]:
def analyze_chunks(name, chunks):
    lengths = [len(chunk.page_content) for chunk in chunks]

    print(f"\n{name}")
    print("-" * 40)
    print(f"Number of chunks : {len(chunks)}")
    print(f"Min size         : {min(lengths)}")
    print(f"Max size         : {max(lengths)}")
    print(f"Average size     : {sum(lengths) / len(lengths):.2f}")

analyze_chunks("Small", small_chunks)
analyze_chunks("Medium", chunks)
analyze_chunks("Large", large_chunks)


Small
----------------------------------------
Number of chunks : 183
Min size         : 59
Max size         : 199
Average size     : 149.19

Medium
----------------------------------------
Number of chunks : 70
Min size         : 118
Max size         : 499
Average size     : 400.46

Large
----------------------------------------
Number of chunks : 37
Min size         : 130
Max size         : 999
Average size     : 762.68


## Observations

### Chunk Size

- Smaller chunk sizes create more chunks.
- Smaller chunks can provide more focused information but may lose context.
- Larger chunks preserve more context but may contain unrelated information.
- `chunk_size` is a target size rather than a guarantee that every chunk will have exactly that length.

### Chunk Overlap

- Overlap preserves information across chunk boundaries.
- Increasing overlap generally increases redundancy and chunk count.
- Too little overlap can break related information across chunks.
- Too much overlap can create unnecessary duplication.

### Recursive Character Splitting

`RecursiveCharacterTextSplitter` attempts to preserve meaningful
text boundaries before falling back to smaller separators.

The best chunk configuration depends on the document and retrieval
task. It should ultimately be evaluated based on retrieval quality,
not only chunk count or chunk size.